# LeetCode 807 — Max Increase to Keep City Skyline bằng Python

Link đề: https://leetcode.com/problems/max-increase-to-keep-city-skyline/ · Độ khó: **Medium** · Chủ đề: Greedy, Array, Matrix

Skeleton LeetCode:

```python
class Solution:
    def maxIncreaseKeepingSkyline(self, grid: List[List[int]]) -> int:
        ...
```

Ghi chú: bài có **đáp án duy nhất** (một số nguyên) nên so sánh bằng `==`. Điểm đặc biệt: lời giải tham lam "mỗi ô tăng tới `min(max hàng, max cột)`" nghe hợp lý nhưng cần **chứng minh** (tăng đồng thời nhiều ô vẫn giữ đường chân trời). Notebook kiểm chứng bằng **oracle vét cạn thật sự**: liệt kê *mọi* lưới mới trên các lưới nhỏ rồi chọn tổng lớn nhất.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: thành phố `n × n` ô, mỗi ô một toà nhà hình hộp; **đường chân trời (skyline)** nhìn từ 4 hướng (bắc, nam, đông, tây) là các **giá trị lớn nhất theo cột / theo hàng**; tăng chiều cao các toà nhà sao cho **không đổi** skyline và **tổng mức tăng lớn nhất**.
2. Nắm các công cụ Python: ma trận list-of-list, `max` theo hàng, **chuyển vị** bằng `zip(*grid)`, list comprehension hai tầng, `sum` kép, và cạm bẫy "iterator chỉ dùng được một lần".
3. Đi từ **thử từng ô tăng dần tới khi skyline đổi (brute force)** → **tính max hàng, max cột rồi lấy `min` (tham lam, hai lượt)** → **cách viết "Pythonic" bằng `zip`** → **dựng lưới mới rồi kiểm chứng skyline**.
4. Hiểu mấu chốt: ô `(i, j)` không được vượt `rowMax[i]` (giữ skyline đông/tây) và không được vượt `colMax[j]` (giữ skyline bắc/nam), nên **trần** của nó là `min(rowMax[i], colMax[j])`; và **mọi ô cùng chạm trần một lúc** vẫn giữ nguyên skyline.
5. Tránh các lỗi kinh điển: dùng `max` thay cho `min`, nhầm chỉ số `rowMax[j]`/`colMax[i]` (lỗi tinh vi: tổng vẫn đúng nhưng lưới mới thì sai), cộng tổng chiều cao mới thay vì mức tăng, chỉ xét hàng (quên cột), và dùng iterator `zip`/`map` hai lần.

## 2. Đề bài & ví dụ

Một thành phố gồm `n × n` khối; mỗi khối có một toà nhà dạng lăng trụ đứng. Cho ma trận `grid` kích thước `n × n` (chỉ số từ 0), `grid[r][c]` là **chiều cao** toà nhà ở hàng `r`, cột `c`.

**Skyline** của thành phố là đường viền ngoài tạo bởi mọi toà nhà khi nhìn từ xa vào một cạnh của thành phố. Skyline nhìn từ 4 hướng bắc, đông, nam, tây có thể khác nhau.

Ta được **tăng chiều cao của bất kỳ số toà nhà nào, mỗi toà một lượng bất kỳ** (kể cả toà cao `0`). Nhưng việc tăng **không được làm đổi skyline** nhìn từ **bất kỳ** hướng nào.

Trả về **tổng lớn nhất** các mức tăng chiều cao mà không làm đổi skyline.

**Ví dụ 1:** `grid = [[3,0,8,4],[2,4,5,7],[9,2,6,3],[0,3,1,0]]` → `35`. Lưới mới (một cách tối ưu):

```
[[8, 4, 8, 7],
 [7, 4, 7, 7],
 [9, 4, 8, 7],
 [3, 3, 3, 3]]
```

**Ví dụ 2:** `grid = [[0,0,0],[0,0,0],[0,0,0]]` → `0` (tăng bất kỳ toà nào cũng làm đổi skyline).

**Ràng buộc:**

- `n == grid.length == grid[r].length`
- `2 <= n <= 50`
- `0 <= grid[r][c] <= 100`

In [1]:
import random
import time
from itertools import product
from typing import List

grid1 = [[3, 0, 8, 4], [2, 4, 5, 7], [9, 2, 6, 3], [0, 3, 1, 0]]
grid2 = [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
new1 = [[8, 4, 8, 7], [7, 4, 7, 7], [9, 4, 8, 7], [3, 3, 3, 3]]


def show(grid, title=None):
    if title:
        print(title)
    for row in grid:
        print(" ".join(f"{v:3d}" for v in row))


show(grid1, "grid ví dụ 1:")
show(new1, "lưới mới (ví dụ 1):")
print("tổng mức tăng =", sum(new1[i][j] - grid1[i][j] for i in range(4) for j in range(4)))

grid ví dụ 1:
  3   0   8   4
  2   4   5   7
  9   2   6   3
  0   3   1   0
lưới mới (ví dụ 1):
  8   4   8   7
  7   4   7   7
  9   4   8   7
  3   3   3   3
tổng mức tăng = 35


## 3. Kiến thức nền

### 3.1 Skyline = max theo hàng và theo cột

Nhìn từ **tây** hoặc **đông** vào thành phố, mỗi *hàng* cho ta một "thanh" cao bằng toà cao nhất của hàng đó: skyline đông/tây là danh sách `rowMax[i] = max(grid[i])`. Nhìn từ **bắc** hoặc **nam**, mỗi *cột* cho một thanh cao bằng toà cao nhất của cột: skyline bắc/nam là `colMax[j] = max(grid[r][j] for r in range(n))`. Skyline không đổi ⇔ cả `rowMax` và `colMax` không đổi.

In [2]:
def skyline(grid):
    """(skyline bắc/nam = max theo cột, skyline đông/tây = max theo hàng)."""
    n = len(grid)
    col_max = [max(grid[r][c] for r in range(n)) for c in range(n)]
    row_max = [max(row) for row in grid]
    return col_max, row_max


col_max, row_max = skyline(grid1)
print("max theo hàng (đông/tây):", row_max)
print("max theo cột (bắc/nam)  :", col_max)
print("skyline lưới mới       :", skyline(new1), "-> giống:", skyline(new1) == skyline(grid1))

max theo hàng (đông/tây): [8, 7, 9, 3]
max theo cột (bắc/nam)  : [9, 4, 8, 7]
skyline lưới mới       : ([9, 4, 8, 7], [8, 7, 9, 3]) -> giống: True


### 3.2 Chuyển vị và `zip(*grid)`

Phép **chuyển vị** đổi hàng thành cột. Trong Python: `list(zip(*grid))` — dấu `*` "mở" các hàng thành các đối số riêng của `zip`, `zip` ghép phần tử cùng vị trí của mỗi hàng, tức mỗi bộ ra là **một cột**. Vì vậy `col_max = [max(col) for col in zip(*grid)]`.

In [3]:
g = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
print(list(zip(*g)))
print([max(col) for col in zip(*g)], [max(row) for row in g])

[(1, 4, 7), (2, 5, 8), (3, 6, 9)]
[7, 8, 9] [3, 6, 9]


### 3.3 Cạm bẫy: `zip`, `map` là iterator dùng **một lần**

`zip(*g)` và `map(max, ...)` trả về **iterator**: duyệt xong là **cạn**. Nếu dùng lại (ví dụ trong vòng lặp lồng nhau) thì lần sau thấy rỗng, và code chạy **không lỗi** nhưng sai. Muốn dùng nhiều lần phải `list(...)` hoặc `tuple(...)`.

In [4]:
cols = zip(*g)
print("lần 1:", list(cols))
print("lần 2:", list(cols), "<- iterator đã cạn!")
col_max_iter = map(max, zip(*g))
print([x for x in col_max_iter], [x for x in col_max_iter])
col_max_list = list(map(max, zip(*g)))
print(col_max_list, col_max_list)

lần 1: [(1, 4, 7), (2, 5, 8), (3, 6, 9)]
lần 2: [] <- iterator đã cạn!
[7, 8, 9] []
[7, 8, 9] [7, 8, 9]


### 3.4 Tổng kép bằng list comprehension

`sum(f(i, j) for i in range(n) for j in range(n))` cộng một biểu thức trên mọi ô. Với tham lam bài này: `sum(min(row_max[i], col_max[j]) - grid[i][j] for i in range(n) for j in range(n))`.

### 3.5 Vì sao cần "vét cạn thật" để tin tham lam

Với bài tham lam, câu hỏi quan trọng là: *tối ưu từng ô riêng lẻ có cho tối ưu toàn cục không?* (có thể việc tăng ô này **ngăn** ô khác). Ở đây các ràng buộc của các ô **độc lập** (mỗi ô chỉ cần `≤ min(rowMax, colMax)`) và đạt đồng thời — ta chứng minh ở mục 10 và kiểm chứng bằng cách **liệt kê mọi lưới mới có thể** trên lưới nhỏ.

## 4. Ý tưởng nền tảng / trực giác

Nhìn ví dụ 1 theo hàng và cột:

```
          cột0 cột1 cột2 cột3 | max hàng
hàng0       3    0    8    4  |   8
hàng1       2    4    5    7  |   7
hàng2       9    2    6    3  |   9
hàng3       0    3    1    0  |   3
-------------------------------
max cột      9    4    8    7
```

Ô `(0, 0)` cao `3`. Nếu ta xây nó lên `8`: hàng 0 có max là `8` → vẫn `8` (không đổi). Nhưng cột 0 có max `9` → `8 ≤ 9` cũng không đổi. Có thể lên tới `9`? Hàng 0 có max `8`, mà `9 > 8` sẽ làm skyline đông/tây hàng 0 đổi thành `9`. Vậy trần của ô `(0,0)` là `min(8, 9) = 8`.

Tổng quát: **ô `(i, j)` không thể cao hơn `rowMax[i]` (đông/tây) cũng không cao hơn `colMax[j]` (bắc/nam)**, nên trần của nó là `min(rowMax[i], colMax[j])`. Câu hỏi còn lại: nâng **mọi** ô lên trần của nó có làm đổi skyline không? Không: mỗi hàng có một toà đang đạt `rowMax[i]` và (như ta chứng minh ở mục 10) toà đó sau khi nâng vẫn cao `rowMax[i]` — do đó skyline giữ nguyên.

In [5]:
n = len(grid1)
ceil_grid = [[min(row_max[i], col_max[j]) for j in range(n)] for i in range(n)]
show(ceil_grid, "trần của từng ô = min(max hàng, max cột):")
show([[ceil_grid[i][j] - grid1[i][j] for j in range(n)] for i in range(n)], "mức tăng từng ô:")
print("tổng:", sum(ceil_grid[i][j] - grid1[i][j] for i in range(n) for j in range(n)))
print("skyline sau khi nâng mọi ô lên trần == skyline cũ:", skyline(ceil_grid) == skyline(grid1))

trần của từng ô = min(max hàng, max cột):
  8   4   8   7
  7   4   7   7
  9   4   8   7
  3   3   3   3
mức tăng từng ô:
  5   4   0   3
  5   0   2   0
  0   2   2   4
  3   0   2   3
tổng: 35
skyline sau khi nâng mọi ô lên trần == skyline cũ: True


## 5. Cách 1 — Brute Force: thử tăng từng ô tới khi skyline đổi

**Ý tưởng "ngây thơ":** với từng ô, thử tăng chiều cao `h = g, g+1, g+2, …` (giữ nguyên các ô khác) và **tính lại skyline** của cả thành phố; ngừng ngay khi skyline đổi. Mức tăng tối đa của ô là `h_max − g`. Cộng các mức tăng của mọi ô.

Cách này phụ thuộc **giả thiết** rằng các ô độc lập (tăng ô này tối đa xong rồi vẫn tăng được các ô khác tối đa). Ta sẽ kiểm chứng giả thiết bằng oracle mạnh hơn ở mục 12.

In [6]:
def max_increase_brute(grid, cap=100):
    n = len(grid)
    base = skyline(grid)
    total = 0
    for i in range(n):
        for j in range(n):
            g = grid[i][j]
            best = g
            h = g
            while h + 1 <= cap:
                h += 1
                grid[i][j] = h
                if skyline(grid) != base:
                    break
                best = h
            grid[i][j] = g                       # trả lại chiều cao ban đầu
            total += best - g
    return total


import copy
print(max_increase_brute(copy.deepcopy(grid1)), max_increase_brute(copy.deepcopy(grid2)))
assert max_increase_brute(copy.deepcopy(grid1)) == 35 and max_increase_brute(copy.deepcopy(grid2)) == 0

35 0


**Độ phức tạp:** với `n × n` ô, mỗi ô thử tới `H` mức (`H ≤ 100`), mỗi lần tính skyline `O(n²)`: tổng `O(n⁴ · H)` — với `n = 50` là `~6·10^8` phép: quá chậm. Bộ nhớ `O(n)`.

**Nút thắt:** ta *tính lại skyline* mỗi lần thử, trong khi skyline gốc đã cố định — chỉ cần so sánh ô đang xét với `rowMax[i]` và `colMax[j]` của skyline gốc.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Tính skyline gốc một lần:** `rowMax[i]`, `colMax[j]` (hai lượt duyệt, `O(n²)`).
2. **Trần của từng ô:** `min(rowMax[i], colMax[j])` (ô tăng vượt `rowMax[i]` thì skyline hàng đổi; vượt `colMax[j]` thì skyline cột đổi).
3. **Tổng mức tăng:** `Σ (min(rowMax[i], colMax[j]) − grid[i][j])`.

Ý quan trọng: ta nâng **mọi** ô lên trần **cùng lúc** mà skyline vẫn giữ nguyên (chứng minh ở mục 10), nên tổng này **đạt được**; và vì mỗi ô riêng lẻ không thể vượt trần, nó cũng **tối ưu**.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm: **Cách 2** hai lượt `rowMax`/`colMax` (chuẩn), **Cách 3** viết gọn bằng `zip` / comprehension, **Cách 4** dựng lưới mới rồi **kiểm chứng** skyline.

### Cách 2 — Hai lượt: `rowMax`, `colMax` rồi tổng

**Giải thích từng đoạn:**

1. Lượt 1: `row_max[i] = max(grid[i])`, `col_max[j] = max(cột j)`.
2. Lượt 2: với mọi ô cộng `min(row_max[i], col_max[j]) − grid[i][j]`.

In [7]:
def max_increase_two_pass(grid):
    n = len(grid)
    row_max = [0] * n
    col_max = [0] * n
    for i in range(n):                      # lượt 1: max theo hàng và cột
        for j in range(n):
            row_max[i] = max(row_max[i], grid[i][j])
            col_max[j] = max(col_max[j], grid[i][j])
    ans = 0
    for i in range(n):                      # lượt 2: cộng mức tăng của từng ô
        for j in range(n):
            ans += min(row_max[i], col_max[j]) - grid[i][j]
    return ans


print(max_increase_two_pass(grid1), max_increase_two_pass(grid2))
assert max_increase_two_pass(grid1) == 35 and max_increase_two_pass(grid2) == 0

35 0


In [8]:
# Code nộp LeetCode cho Cách 2 (đáp án chính)
class SolutionTwoPass:
    def maxIncreaseKeepingSkyline(self, grid: List[List[int]]) -> int:
        n = len(grid)
        row_max = [0] * n
        col_max = [0] * n
        for i in range(n):
            for j in range(n):
                row_max[i] = max(row_max[i], grid[i][j])
                col_max[j] = max(col_max[j], grid[i][j])
        ans = 0
        for i in range(n):
            for j in range(n):
                ans += min(row_max[i], col_max[j]) - grid[i][j]
        return ans

**Độ phức tạp:** `O(n²)` thời gian (hai lượt qua `n²` ô), `O(n)` bộ nhớ phụ. Đây là tối ưu về bậc vì phải đọc mọi ô ít nhất một lần.

### Cách 3 — Viết gọn bằng `zip(*grid)` và comprehension

**Ý tưởng:** cùng thuật toán nhưng dùng công cụ Python: `row_max = list(map(max, grid))`, `col_max = list(map(max, zip(*grid)))` (**bọc `list`** vì `map` trả iterator dùng một lần!), rồi một tổng kép.

In [9]:
def max_increase_zip(grid):
    row_max = list(map(max, grid))               # max từng hàng
    col_max = list(map(max, zip(*grid)))         # max từng cột (zip(*grid) là các cột); bọc list!
    # tổng các trần Σ min(rowMax[i], colMax[j]) trừ tổng chiều cao ban đầu
    return sum(min(r, c) for r in row_max for c in col_max) - sum(map(sum, grid))


print(max_increase_zip(grid1), max_increase_zip(grid2))
assert max_increase_zip(grid1) == 35 and max_increase_zip(grid2) == 0

35 0


Điểm tinh tế: ta viết đáp án là `Σ_{i,j} min(rowMax[i], colMax[j]) − Σ grid` — tổng các trần trừ tổng chiều cao ban đầu. Vì mọi cặp `(i, j)` xuất hiện đúng một lần nên `for r in row_max for c in col_max` duyệt đúng các cặp `(rowMax[i], colMax[j])` mà không cần chỉ số ô.

In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionZip:
    def maxIncreaseKeepingSkyline(self, grid: List[List[int]]) -> int:
        row_max = list(map(max, grid))
        col_max = list(map(max, zip(*grid)))
        return sum(min(r, c) for r in row_max for c in col_max) - sum(map(sum, grid))

**Độ phức tạp:** `O(n²)` thời gian; `zip(*grid)` tạo `n` bộ nên bộ nhớ phụ `O(n)` (iterator) — nhưng nếu `list(zip(*grid))` thì `O(n²)`.

### Cách 4 — Dựng lưới mới rồi kiểm chứng skyline

**Ý tưởng khác:** đừng chỉ tin công thức; hãy **dựng** lưới mới `new[i][j] = min(rowMax[i], colMax[j])`, **kiểm tra** bằng code rằng skyline của `new` bằng skyline của `grid` và `new ≥ grid` cho từng ô, rồi trả `Σ(new − grid)`. Đây là cách "đáp án có chứng nhận": nếu kiểm tra thất bại thì ta biết ngay có lỗi lý luận.

In [11]:
def max_increase_construct(grid, verify=True):
    n = len(grid)
    row_max = [max(row) for row in grid]
    col_max = [max(grid[i][j] for i in range(n)) for j in range(n)]
    new = [[min(row_max[i], col_max[j]) for j in range(n)] for i in range(n)]
    if verify:
        assert all(new[i][j] >= grid[i][j] for i in range(n) for j in range(n)), "có ô bị giảm"
        assert skyline(new) == skyline(grid), "skyline bị đổi"
    return sum(new[i][j] - grid[i][j] for i in range(n) for j in range(n))


print(max_increase_construct(grid1), max_increase_construct(grid2))
assert max_increase_construct(grid1) == 35 and max_increase_construct(grid2) == 0

35 0


In [12]:
# Code nộp LeetCode cho Cách 4 (có kiểm chứng đã bỏ ở bản nộp, chỉ dựng lưới mới)
class SolutionConstruct:
    def maxIncreaseKeepingSkyline(self, grid: List[List[int]]) -> int:
        n = len(grid)
        row_max = [max(row) for row in grid]
        col_max = [max(grid[i][j] for i in range(n)) for j in range(n)]
        new = [[min(row_max[i], col_max[j]) for j in range(n)] for i in range(n)]
        return sum(new[i][j] - grid[i][j] for i in range(n) for j in range(n))

**Độ phức tạp:** `O(n²)` thời gian, `O(n²)` bộ nhớ (lưới mới). Bản "có kiểm chứng" tốn gấp vài lần nhưng cho bằng chứng chạy thực tế.

## 8. TRACE từng bước

**Trace Cách 2** trên ví dụ 1. Lượt 1 cập nhật `row_max` và `col_max` theo từng ô; lượt 2 cộng mức tăng. Bảng lượt 1 (chỉ ghi những lần `row_max`/`col_max` thực sự thay đổi):

| ô `(i, j)` | `grid` | `row_max[i]` sau | `col_max[j]` sau |
|---|---|---|---|
| `(0,0)` | 3 | 3 | 3 |
| `(0,1)` | 0 | 3 | 0 |
| `(0,2)` | 8 | 8 | 8 |
| `(0,3)` | 4 | 8 | 4 |
| `(1,0)` | 2 | 2 | 3 (giữ) |
| ... | ... | ... | ... |

Cuối lượt 1: `row_max = [8, 7, 9, 3]`, `col_max = [9, 4, 8, 7]`. Lượt 2 (từng ô): `(0,0)`: `min(8, 9) − 3 = 5`; `(0,1)`: `min(8, 4) − 0 = 4`; `(0,2)`: `min(8, 8) − 8 = 0`; ... Hàm trace in từng bước đầy đủ:

In [13]:
def trace_two_pass(grid):
    n = len(grid)
    row_max = [0] * n
    col_max = [0] * n
    print("Lượt 1: cập nhật max hàng / max cột")
    for i in range(n):
        for j in range(n):
            changed = []
            if grid[i][j] > row_max[i]:
                row_max[i] = grid[i][j]
                changed.append(f"row_max[{i}]={row_max[i]}")
            if grid[i][j] > col_max[j]:
                col_max[j] = grid[i][j]
                changed.append(f"col_max[{j}]={col_max[j]}")
            print(f"  ({i},{j}) cao {grid[i][j]:2d}: " + (", ".join(changed) if changed else "không đổi"))
    print("row_max =", row_max, "| col_max =", col_max)
    print("Lượt 2: mức tăng từng ô = min(row_max[i], col_max[j]) - grid[i][j]")
    ans = 0
    for i in range(n):
        parts = []
        for j in range(n):
            inc = min(row_max[i], col_max[j]) - grid[i][j]
            ans += inc
            parts.append(f"min({row_max[i]},{col_max[j]})-{grid[i][j]}={inc}")
        print(f"  hàng {i}: " + " | ".join(parts))
    print("=> tổng", ans)
    return ans


assert trace_two_pass(grid1) == 35

Lượt 1: cập nhật max hàng / max cột
  (0,0) cao  3: row_max[0]=3, col_max[0]=3
  (0,1) cao  0: không đổi
  (0,2) cao  8: row_max[0]=8, col_max[2]=8
  (0,3) cao  4: col_max[3]=4
  (1,0) cao  2: row_max[1]=2
  (1,1) cao  4: row_max[1]=4, col_max[1]=4
  (1,2) cao  5: row_max[1]=5
  (1,3) cao  7: row_max[1]=7, col_max[3]=7
  (2,0) cao  9: row_max[2]=9, col_max[0]=9
  (2,1) cao  2: không đổi
  (2,2) cao  6: không đổi
  (2,3) cao  3: không đổi
  (3,0) cao  0: không đổi
  (3,1) cao  3: row_max[3]=3
  (3,2) cao  1: không đổi
  (3,3) cao  0: không đổi
row_max = [8, 7, 9, 3] | col_max = [9, 4, 8, 7]
Lượt 2: mức tăng từng ô = min(row_max[i], col_max[j]) - grid[i][j]
  hàng 0: min(8,9)-3=5 | min(8,4)-0=4 | min(8,8)-8=0 | min(8,7)-4=3
  hàng 1: min(7,9)-2=5 | min(7,4)-4=0 | min(7,8)-5=2 | min(7,7)-7=0
  hàng 2: min(9,9)-9=0 | min(9,4)-2=2 | min(9,8)-6=2 | min(9,7)-3=4
  hàng 3: min(3,9)-0=3 | min(3,4)-3=0 | min(3,8)-1=2 | min(3,7)-0=3
=> tổng 35


Một trace nhỏ nữa với lưới `2 × 2`: `[[1, 5], [4, 2]]`: `row_max = [5, 4]`, `col_max = [4, 5]`; mức tăng: `(0,0): min(5,4) − 1 = 3`; `(0,1): min(5,5) − 5 = 0`; `(1,0): min(4,4) − 4 = 0`; `(1,1): min(4,5) − 2 = 2`. Tổng `5`.

In [14]:
assert trace_two_pass([[1, 5], [4, 2]]) == 5

Lượt 1: cập nhật max hàng / max cột
  (0,0) cao  1: row_max[0]=1, col_max[0]=1
  (0,1) cao  5: row_max[0]=5, col_max[1]=5
  (1,0) cao  4: row_max[1]=4, col_max[0]=4
  (1,1) cao  2: không đổi
row_max = [5, 4] | col_max = [4, 5]
Lượt 2: mức tăng từng ô = min(row_max[i], col_max[j]) - grid[i][j]
  hàng 0: min(5,4)-1=3 | min(5,5)-5=0
  hàng 1: min(4,4)-4=0 | min(4,5)-2=2
=> tổng 5


## 9. Code nộp LeetCode

Chọn **hai lượt** (rõ ràng, `O(n²)` thời gian, `O(n)` bộ nhớ). Giữ nguyên `class Solution`, `maxIncreaseKeepingSkyline(self, grid)`:

In [15]:
from typing import List


class Solution:
    def maxIncreaseKeepingSkyline(self, grid: List[List[int]]) -> int:
        n = len(grid)
        row_max = [0] * n                 # skyline đông/tây
        col_max = [0] * n                 # skyline bắc/nam
        for i in range(n):
            for j in range(n):
                row_max[i] = max(row_max[i], grid[i][j])
                col_max[j] = max(col_max[j], grid[i][j])
        ans = 0
        for i in range(n):
            for j in range(n):
                ans += min(row_max[i], col_max[j]) - grid[i][j]   # nâng ô lên trần của nó
        return ans


print(Solution().maxIncreaseKeepingSkyline([[3, 0, 8, 4], [2, 4, 5, 7], [9, 2, 6, 3], [0, 3, 1, 0]]))   # 35
print(Solution().maxIncreaseKeepingSkyline([[0, 0, 0], [0, 0, 0], [0, 0, 0]]))                            # 0
assert Solution().maxIncreaseKeepingSkyline(copy.deepcopy(grid1)) == 35
assert Solution().maxIncreaseKeepingSkyline(copy.deepcopy(grid2)) == 0

35
0


## 10. Vì sao đúng

Ký hiệu `R_i = rowMax[i]`, `C_j = colMax[j]` (skyline gốc) và `u_{ij} = min(R_i, C_j)`.

**(a) Cận trên: mọi lưới hợp lệ có `new_{ij} ≤ u_{ij}`.** Nếu `new_{ij} > R_i` thì `rowMax` của hàng `i` trong lưới mới `> R_i`: skyline đông/tây đổi. Tương tự `new_{ij} ≤ C_j`. Vậy `new_{ij} ≤ min(R_i, C_j)`. Ngoài ra `new_{ij} ≥ grid_{ij}` (chỉ được tăng). Do đó tổng mức tăng của lưới hợp lệ bất kỳ `≤ Σ (u_{ij} − grid_{ij})`.

**(b) Đạt được: lưới `u` hợp lệ.**

- `u_{ij} ≥ grid_{ij}` vì `grid_{ij} ≤ R_i` và `grid_{ij} ≤ C_j`.
- Max hàng `i` của `u` bằng `R_i`: mọi `u_{ij} ≤ R_i` nên max `≤ R_i`; ngược lại gọi `j*` là cột có `grid_{ij*} = R_i`. Khi đó `C_{j*} ≥ grid_{ij*} = R_i`, nên `u_{ij*} = min(R_i, C_{j*}) = R_i`. Vậy max hàng đúng bằng `R_i`.
- Max cột `j` của `u` bằng `C_j`: đối xứng (lấy hàng `i*` có `grid_{i*j} = C_j`, khi đó `R_{i*} ≥ C_j` nên `u_{i*j} = C_j`).

Vậy skyline giữ nguyên và tổng mức tăng bằng đúng cận trên ở (a): **tối ưu**. ∎

**(c) Vì sao các ô "độc lập".** Ràng buộc skyline chỉ là `max hàng ≤ R_i` và `max cột ≤ C_j` (cộng điều kiện đạt bằng) nên tập lưới hợp lệ là một **hộp** `{grid ≤ new ≤ u}` giao với điều kiện "mỗi hàng/cột còn ít nhất một ô đạt trần". Điểm `u` — đỉnh trên của hộp — thoả điều kiện đạt trần như (b). Do đó tối ưu từng ô cũng là tối ưu toàn cục, và Cách 1 (brute từng ô) cho cùng đáp án.

Ta kiểm chứng (a) và (b) bằng máy: **liệt kê mọi lưới mới** trên các lưới nhỏ và chọn tổng lớn nhất (mục 12D).

## 11. Time / Space Complexity

Đặt `n` là cạnh lưới (`n² ` ô).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử từng ô tới khi skyline đổi | `O(n⁴ · H)` (`H ≤ 100`) | `O(n)` | chỉ làm oracle cho lưới nhỏ |
| 2. Hai lượt `rowMax`/`colMax` | `O(n²)` | `O(n)` | **đáp án chính** |
| 3. `zip` / comprehension | `O(n²)` | `O(n)` | ngắn nhất; nhớ `list(map(...))` |
| 4. Dựng lưới mới + kiểm chứng | `O(n²)` | `O(n²)` | có chứng nhận chạy thực tế |

Đo thời gian ở `n = 50` (lớn nhất theo đề):

In [16]:
def random_grid(rng, n, hi=100):
    return [[rng.randint(0, hi) for _ in range(n)] for _ in range(n)]


def bench(fn, grid, repeat=20):
    best = 1e9
    for _ in range(repeat):
        g = copy.deepcopy(grid)
        t0 = time.perf_counter()
        fn(g)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(807)
g50 = random_grid(rng, 50)
for name, fn in [("two_pass", max_increase_two_pass), ("zip", max_increase_zip), ("construct", lambda g: max_increase_construct(g, verify=False))]:
    print(f"n = 50  {name:10s} {bench(fn, g50) * 1000:7.3f} ms")
g8 = random_grid(rng, 8, 30)
print(f"n = 8   brute      {bench(max_increase_brute, g8, 3) * 1000:7.1f} ms (đã nhỏ mà vẫn chậm hơn hẳn)")

n = 50  two_pass     0.591 ms
n = 50  zip          0.325 ms
n = 50  construct    0.622 ms


n = 8   brute          8.2 ms (đã nhỏ mà vẫn chậm hơn hẳn)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) **oracle vét cạn thật sự** (liệt kê mọi lưới mới), (E) ngẫu nhiên có seed + tính chất bất biến, (F) giới hạn lớn nhất.

### A. Ví dụ của đề

In [17]:
IMPLS = {
    "brute": max_increase_brute,
    "two_pass": max_increase_two_pass,
    "zip": max_increase_zip,
    "construct": max_increase_construct,
    "Solution": lambda g: Solution().maxIncreaseKeepingSkyline(g),
    "SolutionTwoPass": lambda g: SolutionTwoPass().maxIncreaseKeepingSkyline(g),
    "SolutionZip": lambda g: SolutionZip().maxIncreaseKeepingSkyline(g),
    "SolutionConstruct": lambda g: SolutionConstruct().maxIncreaseKeepingSkyline(g),
}
FAST = [k for k in IMPLS if k != "brute"]


def check(grid, want, names=None):
    for name in (names or IMPLS):
        got = IMPLS[name](copy.deepcopy(grid))
        assert got == want, (name, grid, got, want)


check(grid1, 35)
check(grid2, 0)
print("ví dụ 1, 2: cả", len(IMPLS), "cài đặt đúng")

ví dụ 1, 2: cả 8 cài đặt đúng


### B. Edge case

- Lưới `2 × 2` nhỏ nhất; toàn `0`; toàn một giá trị (không tăng được); toàn `100`.
- Một hàng "cao" và các hàng khác thấp.
- Đúng **một** toà cao nhất toàn cục (nó quyết định cả hàng và cột của nó).
- Lưới đã "đạt trần" mọi ô (ma trận `min(R_i, C_j)` chính nó): mức tăng `0`.
- Lưới đường chéo (`diag`): mỗi hàng/cột chỉ có một toà.
- Giá trị biên `0` và `100`.

In [18]:
# chỉ viết tay các đáp án chắc chắn; các ca khác dùng oracle vét cạn ở mục D
hand = [
    ([[0, 0], [0, 0]], 0),
    ([[5, 5], [5, 5]], 0),
    ([[100, 100], [100, 100]], 0),
    ([[5, 0], [0, 5]], 10),                  # hai ô 0 đều nâng lên 5
    ([[1, 5], [4, 2]], 5),                   # ví dụ trace
    ([[0, 0, 9], [0, 0, 0], [0, 0, 0]], 0),  # hàng 0 max 9 nhưng cột 0,1 max 0: trần các ô khác = min(0 hoặc 9, ...) = 0
]
for grid, want in hand[:5]:
    check(grid, want)
check(hand[5][0], 0)
print("edge case viết tay OK")

# đã đạt trần: lưới u = min(R_i, C_j) có mức tăng 0
rng = random.Random(1)
for _ in range(50):
    g = random_grid(rng, rng.randint(2, 8), 20)
    R, C = [max(r) for r in g], [max(col) for col in zip(*g)]
    u = [[min(R[i], C[j]) for j in range(len(g))] for i in range(len(g))]
    check(u, 0, FAST)
print("lưới đã đạt trần: mức tăng 0 (50 lưới ngẫu nhiên)")

edge case viết tay OK
lưới đã đạt trần: mức tăng 0 (50 lưới ngẫu nhiên)


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [19]:
def bug_max_instead_of_min(grid):                 # dùng max thay vì min
    n = len(grid)
    R = [max(r) for r in grid]
    C = [max(col) for col in zip(*grid)]
    return sum(max(R[i], C[j]) - grid[i][j] for i in range(n) for j in range(n))


def bug_swapped_index(grid):                      # nhầm rowMax[j], colMax[i]
    n = len(grid)
    R = [max(r) for r in grid]
    C = [max(col) for col in zip(*grid)]
    return sum(min(R[j], C[i]) - grid[i][j] for i in range(n) for j in range(n))


def bug_sum_new_heights(grid):                    # cộng chiều cao mới thay vì mức tăng
    n = len(grid)
    R = [max(r) for r in grid]
    C = [max(col) for col in zip(*grid)]
    return sum(min(R[i], C[j]) for i in range(n) for j in range(n))


def bug_row_only(grid):                           # chỉ xét hàng, quên cột
    return sum(max(r) - v for r in grid for v in r)


def bug_global_max(grid):                         # dùng max toàn cục
    top = max(max(r) for r in grid)
    return sum(top - v for r in grid for v in r)


def bug_iterator_reuse(grid):                     # col_max là iterator dùng một lần
    n = len(grid)
    R = [max(r) for r in grid]
    C = map(max, zip(*grid))                      # SAI: không bọc list
    total = 0
    for i in range(n):
        for j, c in enumerate(C):                 # lần i = 0 tiêu thụ hết iterator; các hàng sau thấy rỗng
            total += min(R[i], c) - grid[i][j]
    return total


BUGS = [bug_max_instead_of_min, bug_sum_new_heights, bug_row_only, bug_global_max, bug_iterator_reuse]
rng = random.Random(2)
tests = [random_grid(rng, rng.randint(2, 5), 9) for _ in range(300)]
for bug in BUGS:
    bad = next(g for g in tests if bug(copy.deepcopy(g)) != max_increase_two_pass(g))
    print(f"{bug.__name__:24s} sai trên {bad}: ra {bug(copy.deepcopy(bad))}, đúng {max_increase_two_pass(bad)}")

bug_max_instead_of_min   sai trên [[1, 1], [5, 2]]: ra 8, đúng 0
bug_sum_new_heights      sai trên [[1, 1], [5, 2]]: ra 9, đúng 0
bug_row_only             sai trên [[1, 1], [5, 2]]: ra 3, đúng 0
bug_global_max           sai trên [[1, 1], [5, 2]]: ra 11, đúng 0
bug_iterator_reuse       sai trên [[4, 9, 3, 9], [0, 9, 2, 6], [6, 8, 5, 8], [7, 8, 4, 0]]: ra 5, đúng 28


### D. Oracle vét cạn thật sự: liệt kê mọi lưới mới

**Không dựa vào tham lam:** với lưới gốc `g` nhỏ, liệt kê **mọi** lưới `new` có `new ≥ g` cùng ô, **cùng skyline** (cả max hàng và max cột), rồi lấy tổng mức tăng lớn nhất. Để nhanh, ta gom sẵn mọi lưới theo khoá skyline. Kiểm chứng với:

- mọi lưới `2 × 2` giá trị `0..4` (`5⁴ = 625` lưới);
- mọi lưới `3 × 3` giá trị `0..1` (`512` lưới);
- `1500` lưới `3 × 3` giá trị `0..2` lấy mẫu có seed;
- `1500` lưới `4 × 4` giá trị `0..1` lấy mẫu có seed (trong tổng `65536`).

In [20]:
def true_oracle_factory(n, hi):
    """Trả về hàm oracle(g): max tổng mức tăng bằng liệt kê MỌI lưới mới có cùng skyline và >= g từng ô."""
    groups = {}
    for cells in product(range(hi + 1), repeat=n * n):
        grid = [list(cells[i * n:(i + 1) * n]) for i in range(n)]
        key = skyline(grid)
        groups.setdefault((tuple(key[0]), tuple(key[1])), []).append(cells)

    def oracle(g):
        key = skyline(g)
        flat = [v for row in g for v in row]
        best = None
        for cells in groups[(tuple(key[0]), tuple(key[1]))]:
            if all(c >= f for c, f in zip(cells, flat)):
                gain = sum(cells) - sum(flat)
                if best is None or gain > best:
                    best = gain
        return best

    return oracle, groups


tested = 0
oracle22, groups22 = true_oracle_factory(2, 4)
for cells in product(range(5), repeat=4):
    g = [list(cells[:2]), list(cells[2:])]
    want = oracle22(g)
    check(g, want)
    tested += 1
print("2x2 giá trị 0..4:", tested, "lưới, tất cả cài đặt khớp oracle vét cạn (", sum(len(v) for v in groups22.values()), "lưới mới đã liệt kê )")

oracle33a, _ = true_oracle_factory(3, 1)
cnt = 0
for cells in product(range(2), repeat=9):
    g = [list(cells[i * 3:(i + 1) * 3]) for i in range(3)]
    check(g, oracle33a(g))
    cnt += 1
print("3x3 giá trị 0..1:", cnt, "lưới khớp")

2x2 giá trị 0..4: 625 lưới, tất cả cài đặt khớp oracle vét cạn ( 625 lưới mới đã liệt kê )


3x3 giá trị 0..1: 512 lưới khớp


In [21]:
t0 = time.perf_counter()
oracle33b, groups33 = true_oracle_factory(3, 2)
rng = random.Random(33)
for _ in range(1500):
    g = [[rng.randint(0, 2) for _ in range(3)] for _ in range(3)]
    check(g, oracle33b(g))
print("3x3 giá trị 0..2: 1500 lưới mẫu khớp oracle (", sum(len(v) for v in groups33.values()), "lưới mới được liệt kê; nhóm lớn nhất", max(len(v) for v in groups33.values()), ")")

oracle44, groups44 = true_oracle_factory(4, 1)
rng = random.Random(44)
for _ in range(1500):
    g = [[rng.randint(0, 1) for _ in range(4)] for _ in range(4)]
    check(g, oracle44(g))
print("4x4 giá trị 0..1: 1500 lưới mẫu khớp oracle; tổng thời gian cell:", round(time.perf_counter() - t0, 1), "s")

3x3 giá trị 0..2: 1500 lưới mẫu khớp oracle ( 19683 lưới mới được liệt kê; nhóm lớn nhất 4051 )


4x4 giá trị 0..1: 1500 lưới mẫu khớp oracle; tổng thời gian cell: 17.5 s


### E. Ngẫu nhiên có seed và tính chất bất biến

Sinh lưới ngẫu nhiên đúng ràng buộc (`n` từ `2` đến `50`, giá trị `0..100`) so sánh các cách; thêm các **tính chất** (metamorphic):

1. Chuyển vị lưới không đổi đáp án.
2. Đảo thứ tự các hàng (hoặc cột) không đổi đáp án; xoay `90°` cũng không.
3. Cộng cùng một hằng `+c` vào mọi ô (nếu còn `≤` giới hạn) không đổi đáp án.
4. Đáp án `≥ 0`, và `= 0` khi lưới đã đạt trần.

In [22]:
rng = random.Random(20240807)
count = 0
for trial in range(400):
    n = rng.randint(2, 12)
    hi = rng.choice([1, 3, 10, 100])
    g = random_grid(rng, n, hi)
    want = max_increase_two_pass(g)
    for name in FAST:
        assert IMPLS[name](copy.deepcopy(g)) == want, (name, g)
    count += 1
for trial in range(30):
    g = random_grid(rng, rng.randint(30, 50), 100)
    want = max_increase_two_pass(g)
    for name in FAST:
        assert IMPLS[name](copy.deepcopy(g)) == want
    count += 1
print(count, "lưới ngẫu nhiên (cả n tới 50): mọi cài đặt trùng nhau")

for _ in range(300):
    n = rng.randint(2, 9)
    g = random_grid(rng, n, 50)
    base = Solution().maxIncreaseKeepingSkyline(copy.deepcopy(g))
    assert base >= 0
    transposed = [list(c) for c in zip(*g)]
    assert Solution().maxIncreaseKeepingSkyline(copy.deepcopy(transposed)) == base
    rows = copy.deepcopy(g)
    rng.shuffle(rows)
    assert Solution().maxIncreaseKeepingSkyline(rows) == base
    cols = [list(c) for c in zip(*g)]
    rng.shuffle(cols)
    assert Solution().maxIncreaseKeepingSkyline([list(c) for c in zip(*cols)]) == base
    rotated = [list(r) for r in zip(*g[::-1])]
    assert Solution().maxIncreaseKeepingSkyline(rotated) == base
    shifted = [[v + 25 for v in row] for row in g]
    assert Solution().maxIncreaseKeepingSkyline(shifted) == base
print("tính chất bất biến (chuyển vị, hoán vị hàng/cột, xoay 90°, cộng hằng) OK trên 300 lưới")

430 lưới ngẫu nhiên (cả n tới 50): mọi cài đặt trùng nhau
tính chất bất biến (chuyển vị, hoán vị hàng/cột, xoay 90°, cộng hằng) OK trên 300 lưới


### F. Giới hạn lớn nhất

`n = 50`: lưới ngẫu nhiên `0..100`, lưới toàn `0`, toàn `100`, lưới "bậc thang" `grid[i][j] = i + j` và lưới chỉ có một toà cao `100` ở góc.

In [23]:
n = 50
cases = {
    "ngẫu nhiên": random_grid(random.Random(5), n, 100),
    "toàn 0": [[0] * n for _ in range(n)],
    "toàn 100": [[100] * n for _ in range(n)],
    "bậc thang i+j": [[min(100, i + j) for j in range(n)] for i in range(n)],
    "một toà cao ở góc": [[100 if (i, j) == (0, 0) else 0 for j in range(n)] for i in range(n)],
}
for name, g in cases.items():
    want = sum(min(max(g[i]), max(g[r][j] for r in range(n))) - g[i][j] for i in range(n) for j in range(n))
    check(g, want, FAST)
    print(f"{name:20s}: đáp án {want}")
assert Solution().maxIncreaseKeepingSkyline(copy.deepcopy(cases["toàn 0"])) == 0
assert Solution().maxIncreaseKeepingSkyline(copy.deepcopy(cases["toàn 100"])) == 0
assert Solution().maxIncreaseKeepingSkyline(copy.deepcopy(cases["một toà cao ở góc"])) == 0
print("giới hạn lớn nhất OK")

ngẫu nhiên          : đáp án 120391
toàn 0              : đáp án 0
toàn 100            : đáp án 0
bậc thang i+j       : đáp án 40425
một toà cao ở góc   : đáp án 0
giới hạn lớn nhất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `max` thay vì `min`

Nâng ô lên `max(rowMax, colMax)` làm vượt skyline của hàng hoặc cột còn lại.

In [24]:
g = [[1, 5], [4, 2]]
print("bug_max:", bug_max_instead_of_min(copy.deepcopy(g)), "| đúng:", Solution().maxIncreaseKeepingSkyline(copy.deepcopy(g)))
R, C = [max(r) for r in g], [max(c) for c in zip(*g)]
wrong_new = [[max(R[i], C[j]) for j in range(2)] for i in range(2)]
show(wrong_new, "lưới mới theo max:")
print("skyline gốc:", skyline(g), "| skyline lưới sai:", skyline(wrong_new), "-> đổi!")
assert skyline(wrong_new) != skyline(g)

bug_max: 7 | đúng: 5
lưới mới theo max:
  5   5
  4   5
skyline gốc: ([4, 5], [5, 4]) | skyline lưới sai: ([5, 5], [5, 5]) -> đổi!


**Vì sao sai:** ô `(0,0)` được nâng lên `5` làm cột 0 (max gốc `4`) thành `5`. **Đúng:** trần là `min` của hai skyline.

### Lỗi 2: nhầm chỉ số `rowMax[j]`, `colMax[i]` (lỗi tinh vi)

Hai mảng cùng độ dài `n` nên code chạy không lỗi. Điều bất ngờ: **tổng** `Σ min(R[j], C[i])` trên mọi cặp `(i, j)` bằng `Σ min(R[i], C[j])` (cùng tập các cặp), nên đáp án số **vẫn đúng**! Nhưng **lưới mới** dựng theo chỉ số nhầm thì **sai** (đổi skyline). Lỗi này chỉ lộ ra khi bài đòi trả lưới (bài tập 5) hoặc khi ô được dùng riêng lẻ.

In [25]:
g = [[3, 0, 8, 4], [2, 4, 5, 7], [9, 2, 6, 3], [0, 3, 1, 0]]
print("tổng với chỉ số nhầm:", bug_swapped_index(copy.deepcopy(g)), "| đúng:", Solution().maxIncreaseKeepingSkyline(copy.deepcopy(g)), "(trùng nhau!)")
R, C = [max(r) for r in g], [max(c) for c in zip(*g)]
bad_new = [[min(R[j], C[i]) for j in range(4)] for i in range(4)]
good_new = [[min(R[i], C[j]) for j in range(4)] for i in range(4)]
show(bad_new, "lưới mới dựng với chỉ số nhầm:")
print("skyline gốc :", skyline(g))
print("skyline sai :", skyline(bad_new), "-> đổi")
print("skyline đúng:", skyline(good_new), "-> giữ nguyên")
assert bug_swapped_index(copy.deepcopy(g)) == 35
assert skyline(bad_new) != skyline(g) and skyline(good_new) == skyline(g)

tổng với chỉ số nhầm: 35 | đúng: 35 (trùng nhau!)
lưới mới dựng với chỉ số nhầm:
  8   7   9   3
  4   4   4   3
  8   7   8   3
  7   7   7   3
skyline gốc : ([9, 4, 8, 7], [8, 7, 9, 3])
skyline sai : ([8, 7, 9, 3], [9, 4, 8, 7]) -> đổi
skyline đúng: ([9, 4, 8, 7], [8, 7, 9, 3]) -> giữ nguyên


**Vì sao nguy hiểm:** đáp án số đúng che giấu lý luận sai. **Đúng:** hàng theo `i` (`R[i]`), cột theo `j` (`C[j])`; và luôn **kiểm chứng skyline** của lưới mới khi dựng nó.

### Lỗi 3: cộng chiều cao mới thay vì mức tăng

Đề hỏi **tổng mức tăng**, tức `Σ(mới − cũ)`, không phải `Σ mới`.

In [26]:
print("cộng chiều cao mới:", bug_sum_new_heights(copy.deepcopy(g)), "| đúng (mức tăng):", Solution().maxIncreaseKeepingSkyline(copy.deepcopy(g)), "| tổng chiều cao cũ:", sum(map(sum, g)))
assert bug_sum_new_heights(copy.deepcopy(g)) - sum(map(sum, g)) == 35

cộng chiều cao mới: 92 | đúng (mức tăng): 35 | tổng chiều cao cũ: 57


**Vì sao sai:** thiếu phép trừ chiều cao cũ. **Đúng:** `min(R[i], C[j]) - grid[i][j]`.

### Lỗi 4: chỉ xét hàng, hoặc dùng max toàn cục

Skyline có **hai** ràng buộc (hàng và cột); bỏ một ràng buộc hoặc thay bằng `max` toàn cục cho đáp án quá lớn.

In [27]:
print("chỉ hàng      :", bug_row_only(copy.deepcopy(g)), "| đúng:", Solution().maxIncreaseKeepingSkyline(copy.deepcopy(g)))
print("max toàn cục  :", bug_global_max(copy.deepcopy(g)))
assert bug_row_only(copy.deepcopy(g)) > 35 and bug_global_max(copy.deepcopy(g)) > 35

chỉ hàng      : 51 | đúng: 35
max toàn cục  : 87


**Vì sao sai:** ô ở cột thấp bị nâng quá trần cột. **Đúng:** luôn `min` của **cả hai**.

### Lỗi 5: dùng iterator `map`/`zip` nhiều lần

`C = map(max, zip(*grid))` chỉ duyệt được **một lần**. Trong vòng lặp lồng, các hàng sau thấy `C` rỗng: tổng thiếu, và **không có lỗi nào được báo**.

In [28]:
print("iterator dùng lại:", bug_iterator_reuse(copy.deepcopy(g)), "| đúng:", Solution().maxIncreaseKeepingSkyline(copy.deepcopy(g)))
assert bug_iterator_reuse(copy.deepcopy(g)) != 35

iterator dùng lại: 12 | đúng: 35


**Vì sao sai:** sau hàng `0`, `C` đã cạn nên `enumerate(C)` không lặp. **Đúng:** `C = list(map(max, zip(*grid)))`.

### Lỗi 6 (bonus): tính lại skyline trong lúc sửa lưới

Nếu bạn nâng ô trực tiếp rồi vẫn gọi `max(grid[i])` để lấy skyline cho ô kế tiếp, skyline của bạn đang thay đổi theo lưới. May là ở bài này việc nâng lên đúng trần không đổi max, nhưng thói quen nâng ô *vượt trần* "tạm thời" sẽ gây sai. Luôn tính `rowMax`/`colMax` **một lần từ lưới gốc**.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "tối đa hoá tổng, mỗi phần tử có **trần** do hai hay nhiều ràng buộc độc lập (hàng và cột)" → **tham lam: đặt mỗi phần tử ở trần của nó** (`min` của các trần), sau đó chứng minh tập trần đạt đồng thời.

**Mẹo:**

1. Tính các đại lượng tổng hợp (`rowMax`, `colMax`) **một lần** rồi dùng lại.
2. Mô hình hoá ràng buộc "giữ nguyên skyline" thành "max hàng và max cột không đổi", ra ngay trần `min`.
3. Khi nghi ngờ tham lam, hãy viết **oracle vét cạn trên lưới nhỏ** (liệt kê mọi phương án) như mục 12D.
4. Dùng **tính chất bất biến** để test (chuyển vị, hoán vị hàng/cột, cộng hằng).
5. Cẩn thận iterator dùng một lần (`zip`, `map`, `filter`).
6. Lưới vuông hay chữ nhật: công thức vẫn `min(rowMax[i], colMax[j])`.

**Họ bài cùng pattern:** Projection Area of 3D Shapes (883), Surface Area of 3D Shapes (892), The Skyline Problem (218, khó hơn), Maximum Building Height (1840), Count Servers that Communicate (1267).

## 15. Biến thể

1. **Lưới chữ nhật `m × n`**: công thức giữ nguyên; `zip(*grid)` vẫn cho các cột.
2. **Trả về chính lưới mới** (bài tập 5).
3. **Tổng diện tích chiếu** (Projection Area 883): số ô khác `0` + tổng `rowMax` + tổng `colMax`.
4. **Giảm** chiều cao tối đa mà giữ skyline: mỗi ô chỉ cần giữ nếu nó là "người duy nhất đạt max" của hàng/cột — khó hơn, cần đếm.
5. **Skyline chỉ nhìn từ hai hướng** (chỉ hàng hoặc chỉ cột): chỉ một ràng buộc.

Cài đặt biến thể 1 và 3, đối chiếu với brute:

In [29]:
def max_increase_rect(grid):
    """Biến thể 1: lưới m x n (không nhất thiết vuông)."""
    row_max = [max(r) for r in grid]
    col_max = [max(c) for c in zip(*grid)]
    return sum(min(r, c) for r in row_max for c in col_max) - sum(map(sum, grid))


def skyline_rect(grid):
    return [max(c) for c in zip(*grid)], [max(r) for r in grid]


def brute_rect(grid, cap=20):
    m, n = len(grid), len(grid[0])
    base = skyline_rect(grid)
    total = 0
    for i in range(m):
        for j in range(n):
            g, best, h = grid[i][j], grid[i][j], grid[i][j]
            while h < cap:
                h += 1
                grid[i][j] = h
                if skyline_rect(grid) != base:
                    break
                best = h
            grid[i][j] = g
            total += best - g
    return total


rng = random.Random(15)
for _ in range(200):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    g = [[rng.randint(0, 9) for _ in range(n)] for _ in range(m)]
    assert max_increase_rect(g) == brute_rect(copy.deepcopy(g), 9)
print("biến thể m x n khớp brute trên 200 lưới chữ nhật")


def projection_area(grid):
    """Biến thể 3 (LeetCode 883): diện tích hình chiếu = số ô >0 (nhìn từ trên) + sum(max hàng) + sum(max cột)."""
    return sum(v > 0 for r in grid for v in r) + sum(map(max, grid)) + sum(map(max, zip(*grid)))


print("projection area ví dụ [[1,2],[3,4]]:", projection_area([[1, 2], [3, 4]]), "(đáp án 17)")
assert projection_area([[1, 2], [3, 4]]) == 17 and projection_area([[2]]) == 5

biến thể m x n khớp brute trên 200 lưới chữ nhật
projection area ví dụ [[1,2],[3,4]]: 17 (đáp án 17)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử từng ô | Tăng dần từng ô tới khi skyline đổi | `O(n⁴ · H)` | `O(n)` | Oracle cho lưới nhỏ |
| 2. Hai lượt | `rowMax`, `colMax`, rồi `Σ min − g` | `O(n²)` | `O(n)` | **Đáp án chính**; rõ ràng |
| 3. `zip` / comprehension | Cùng ý, viết gọn; `Σ min − Σ g` | `O(n²)` | `O(n)` | Ngắn gọn; nhớ `list(map(...))` |
| 4. Dựng lưới + kiểm chứng | Dựng `min(R_i, C_j)`, kiểm skyline | `O(n²)` | `O(n²)` | Khi muốn "chứng nhận" chạy thực tế / trả lưới |

Cả bốn khớp nhau và khớp **oracle liệt kê mọi lưới mới** trên hàng nghìn lưới nhỏ (mục 12D).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [30]:
g = [[2, 1, 3], [0, 4, 1], [5, 2, 0]]
print([max(r) for r in g], [max(c) for c in zip(*g)])
print(Solution().maxIncreaseKeepingSkyline(copy.deepcopy(g)))

[3, 4, 5] [5, 4, 3]
14


### Bài 2 (tính tay)

`grid = [[1, 2, 3], [4, 0, 1], [2, 2, 2]]`.

- (a) Tính `rowMax` và `colMax`.
- (b) Viết lưới mới `min(rowMax[i], colMax[j])` và tính tổng mức tăng.
- (c) Ô nào có mức tăng bằng `0` và vì sao?

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện hai lượt:

In [31]:
def increase_todo(grid):
    n = len(grid)
    row_max = [0] * n
    col_max = [0] * n
    for i in range(n):
        for j in range(n):
            pass            # TODO 1: cập nhật row_max[i] và col_max[j] bằng grid[i][j]
    ans = 0
    for i in range(n):
        for j in range(n):
            pass            # TODO 2: cộng min(row_max[i], col_max[j]) - grid[i][j] vào ans
    return ans


# Khi làm xong, bỏ comment để kiểm tra:
# assert increase_todo([[3, 0, 8, 4], [2, 4, 5, 7], [9, 2, 6, 3], [0, 3, 1, 0]]) == 35
# assert increase_todo([[0, 0, 0], [0, 0, 0], [0, 0, 0]]) == 0

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng `zip(*grid)` (nhớ `list`!):

In [32]:
class MySolution:
    def maxIncreaseKeepingSkyline(self, grid):
        pass            # TODO


# Khi làm xong:
# assert MySolution().maxIncreaseKeepingSkyline([[3, 0, 8, 4], [2, 4, 5, 7], [9, 2, 6, 3], [0, 3, 1, 0]]) == 35
# assert MySolution().maxIncreaseKeepingSkyline([[1, 5], [4, 2]]) == 5

### Bài 5 (mini challenge): trả về cả lưới mới

Viết `best_city(grid)` trả về `(tổng mức tăng, lưới mới)` với lưới mới là một cách đạt tối ưu, rồi **tự kiểm chứng** (trong test) rằng lưới mới có cùng skyline, mỗi ô `≥` ô gốc, và tổng khớp. Hỗ trợ cả lưới chữ nhật `m × n`.

In [33]:
def best_city(grid):
    pass            # TODO


# Khi làm xong:
# total, new = best_city([[1, 5], [4, 2]])
# assert total == 5 and new == [[4, 5], [4, 4]]

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `rowMax` theo từng hàng, `colMax` theo từng cột (`zip(*g)`).
- Mức 2: `rowMax = [3, 4, 5]`, `colMax = [5, 4, 3]`.
- Mức 3: trần mỗi ô `min(rowMax[i], colMax[j])`; trừ chiều cao cũ rồi cộng (xem đáp án mục 19).

**Bài 2**

- Mức 1: hàng: `3, 4, 2`; cột: `4, 2, 3`.
- Mức 2: trần: hàng 0: `min(3,4)=3, min(3,2)=2, min(3,3)=3`; hàng 1: `4, 2, 3`; hàng 2: `2, 2, 2`.
- Mức 3: trừ lưới cũ: `[[2,0,0],[0,2,2],[0,0,0]]`, tổng `6`. Ô bằng 0: những ô đã chạm trần (như ô chứa max hàng/cột).

**Bài 3**

- Mức 1: lượt 1 dùng `max(...)` để cập nhật hai mảng.
- Mức 2: `row_max[i] = max(row_max[i], grid[i][j])`.
- Mức 3: lượt 2: `ans += min(row_max[i], col_max[j]) - grid[i][j]`.

**Bài 4**

- Mức 1: `row_max = list(map(max, grid))`.
- Mức 2: `col_max = list(map(max, zip(*grid)))`.
- Mức 3: `return sum(min(r, c) for r in row_max for c in col_max) - sum(map(sum, grid))`.

**Bài 5**

- Mức 1: dựng `new` bằng `min(row_max[i], col_max[j])`.
- Mức 2: tổng = `sum(new) − sum(grid)`.
- Mức 3: để kiểm chứng viết hàm `skyline_rect` (max cột và max hàng) rồi so sánh.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [34]:
g = [[2, 1, 3], [0, 4, 1], [5, 2, 0]]
assert [max(r) for r in g] == [3, 4, 5] and [max(c) for c in zip(*g)] == [5, 4, 3]
ans = Solution().maxIncreaseKeepingSkyline(copy.deepcopy(g))
assert ans == max_increase_brute(copy.deepcopy(g), 10)
print("Bài 1: đáp án =", ans)
assert ans == sum(min(r, c) for r in [3, 4, 5] for c in [5, 4, 3]) - sum(map(sum, g))

Bài 1: đáp án = 14


### Bài 2

(a) `rowMax = [3, 4, 2]`, `colMax = [4, 2, 3]`. (b) Lưới mới `[[3,2,3],[4,2,3],[2,2,2]]`, mức tăng `[[2,0,0],[0,2,2],[0,0,0]]`, tổng `6`. (c) Mức tăng `0` ở các ô đã chạm trần, ví dụ `(0,1)`: `grid = 2 = min(3, 2)`.

In [35]:
g = [[1, 2, 3], [4, 0, 1], [2, 2, 2]]
R, C = [max(r) for r in g], [max(c) for c in zip(*g)]
assert R == [3, 4, 2] and C == [4, 2, 3]
new = [[min(R[i], C[j]) for j in range(3)] for i in range(3)]
assert new == [[3, 2, 3], [4, 2, 3], [2, 2, 2]]
inc = [[new[i][j] - g[i][j] for j in range(3)] for i in range(3)]
assert inc == [[2, 0, 0], [0, 2, 2], [0, 0, 0]] and sum(map(sum, inc)) == 6
assert Solution().maxIncreaseKeepingSkyline(copy.deepcopy(g)) == 6 == max_increase_brute(copy.deepcopy(g), 10)
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [36]:
def increase_answer(grid):
    n = len(grid)
    row_max = [0] * n
    col_max = [0] * n
    for i in range(n):
        for j in range(n):
            row_max[i] = max(row_max[i], grid[i][j])
            col_max[j] = max(col_max[j], grid[i][j])
    ans = 0
    for i in range(n):
        for j in range(n):
            ans += min(row_max[i], col_max[j]) - grid[i][j]
    return ans


assert increase_answer(grid1) == 35 and increase_answer(grid2) == 0
rng = random.Random(3)
for _ in range(200):
    g = random_grid(rng, rng.randint(2, 7), 20)
    assert increase_answer(g) == max_increase_two_pass(g)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def maxIncreaseKeepingSkyline(self, grid):
        row_max = list(map(max, grid))
        col_max = list(map(max, zip(*grid)))
        return sum(min(r, c) for r in row_max for c in col_max) - sum(map(sum, grid))


assert MySolutionAnswer().maxIncreaseKeepingSkyline(copy.deepcopy(grid1)) == 35
assert MySolutionAnswer().maxIncreaseKeepingSkyline([[1, 5], [4, 2]]) == 5
rng = random.Random(4)
for _ in range(200):
    g = random_grid(rng, rng.randint(2, 7), 20)
    assert MySolutionAnswer().maxIncreaseKeepingSkyline(g) == max_increase_two_pass(g)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [38]:
def best_city_answer(grid):
    row_max = [max(r) for r in grid]
    col_max = [max(c) for c in zip(*grid)]
    new = [[min(row_max[i], col_max[j]) for j in range(len(grid[0]))] for i in range(len(grid))]
    total = sum(map(sum, new)) - sum(map(sum, grid))
    return total, new


total, new = best_city_answer([[1, 5], [4, 2]])
assert total == 5 and new == [[4, 5], [4, 4]]
rng = random.Random(5)
for _ in range(300):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    g = [[rng.randint(0, 9) for _ in range(n)] for _ in range(m)]
    total, new = best_city_answer(g)
    assert skyline_rect(new) == skyline_rect(g)                              # cùng skyline
    assert all(new[i][j] >= g[i][j] for i in range(m) for j in range(n))      # chỉ tăng
    assert total == sum(new[i][j] - g[i][j] for i in range(m) for j in range(n)) == brute_rect(copy.deepcopy(g), 9)
print("Bài 5 OK: lưới mới hợp lệ, tổng khớp brute trên 300 lưới chữ nhật")

Bài 5 OK: lưới mới hợp lệ, tổng khớp brute trên 300 lưới chữ nhật


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (tham lam với trần `min` của hai ràng buộc):**

```python
class Solution:
    def maxIncreaseKeepingSkyline(self, grid):
        row_max = [max(r) for r in grid]                 # skyline đông/tây
        col_max = [max(c) for c in zip(*grid)]           # skyline bắc/nam
        return sum(min(r, c) for r in row_max for c in col_max) - sum(map(sum, grid))
```

(hoặc dạng hai lượt: `ans += min(row_max[i], col_max[j]) - grid[i][j]`.)

**Checklist trước khi submit:**

- [ ] `rowMax` theo **hàng**, `colMax` theo **cột**, dùng đúng chỉ số `[i]` cho hàng, `[j]` cho cột?
- [ ] Trần là `min(rowMax[i], colMax[j])` (không phải `max`)?
- [ ] Cộng **mức tăng** (`mới − cũ`), không cộng chiều cao mới?
- [ ] Nếu dùng `zip(*grid)`/`map`: bọc `list(...)` khi dùng lại nhiều lần?
- [ ] Đã thử `[[0,0],[0,0]]` (0), ví dụ 1 (35), lưới đường chéo?
- [ ] Đúng tên `class Solution`, `maxIncreaseKeepingSkyline(self, grid)`, không in debug.